# Rate scenarios

Parallel shifts and key-rate twists on discount (and optional forward) curves: steepener, flattener, and inversion-style shapes. Built-ins are used when present; custom `build_scenario_spec` operations fill the gaps.

## Concept

- **Parallel shift** moves all zero (or par-equivalent) rates by the same amount; discount factors respond uniformly across maturities.
- **Steepener** lifts long-end rates relative to the short end (or the reverse for a **flattener**).
- **Inversion** is a stylized shape where short policy rates sit above long forwards—here we approximate it with **node shocks** that lift the front end and ease the belly/long end.

Operations use `curve_parallel_bp` and `curve_node_bp` with `curve_kind: discount` (and `forward` where shown). Validation runs through `validate_scenario_spec`.

In [ ]:
import json
from datetime import date

from finstack.scenarios import (
    list_builtin_templates,
    build_from_template,
    list_template_components,
    build_template_component,
    build_scenario_spec,
    compose_scenarios,
    apply_scenario_to_market,
    validate_scenario_spec,
)

templates = list_builtin_templates()
print("Built-in templates (rate-related may include rate_shock_2022):", templates)

rate_tpl_json = None
try:
    if "rate_shock_2022" in templates:
        rate_tpl_json = build_from_template("rate_shock_2022")
        print("Loaded template rate_shock_2022; spec length:", len(rate_tpl_json))
        comps = list_template_components("rate_shock_2022")
        print("Components:", comps)
        if comps:
            c0 = build_template_component("rate_shock_2022", comps[0])
            print("First component spec length:", len(c0))
    else:
        print("Template rate_shock_2022 not in this build; relying on custom ops only.")
except Exception as e:
    print("Template branch skipped:", e)

def op_parallel_100():
    return [
        {
            "kind": "curve_parallel_bp",
            "curve_kind": "discount",
            "curve_id": "USD-OIS",
            "discount_curve_id": None,
            "bp": 100.0,
        }
    ]

def op_steepener():
    # Short end ~flat, long end higher (bear steepener in rate space).
    return [
        {
            "kind": "curve_node_bp",
            "curve_kind": "discount",
            "curve_id": "USD-OIS",
            "discount_curve_id": None,
            "nodes": [["6M", 0.0], ["5Y", 60.0]],
            "match_mode": "interpolate",
        }
    ]

def op_flattener():
    # Short end up, long end ~unchanged.
    return [
        {
            "kind": "curve_node_bp",
            "curve_kind": "discount",
            "curve_id": "USD-OIS",
            "discount_curve_id": None,
            "nodes": [["6M", 55.0], ["5Y", 5.0]],
            "match_mode": "interpolate",
        }
    ]

def op_inversion_shape():
    # Stylized inversion: front end up strongly, long end down.
    return [
        {
            "kind": "curve_node_bp",
            "curve_kind": "discount",
            "curve_id": "USD-OIS",
            "discount_curve_id": None,
            "nodes": [["3M", 90.0], ["2Y", 40.0], ["5Y", -25.0]],
            "match_mode": "interpolate",
        }
    ]

specs = {
    "parallel_plus_100bp": build_scenario_spec(
        "parallel_plus_100bp",
        json.dumps(op_parallel_100()),
        "+100bp parallel",
        "Uniform +100bp on USD-OIS discount curve",
    ),
    "steepener": build_scenario_spec(
        "steepener",
        json.dumps(op_steepener()),
        "Steepener",
        "Short flat, long rates up (key-rate style)",
    ),
    "flattener": build_scenario_spec(
        "flattener",
        json.dumps(op_flattener()),
        "Flattener",
        "Short up, long anchored",
    ),
    "inversion_style": build_scenario_spec(
        "inversion_style",
        json.dumps(op_inversion_shape()),
        "Inversion-style twist",
        "Front end up, long end down (illustrative)",
    ),
}

for sid, sj in specs.items():
    ok = validate_scenario_spec(sj)
    print(f"Scenario {sid!r} valid={ok}; json chars={len(sj)}")

merged = compose_scenarios(
    json.dumps([json.loads(specs["parallel_plus_100bp"]), json.loads(specs["steepener"])])
)
print(
    "Composed parallel+steepener:",
    validate_scenario_spec(merged),
    "id=",
    json.loads(merged).get("id"),
    "ops=",
    len(json.loads(merged).get("operations", [])),
)
print("apply_scenario_to_market callable:", callable(apply_scenario_to_market))

if rate_tpl_json:
    print("Historical template snippet:", rate_tpl_json[:240], "...")


**`match_mode: interpolate` on `curve_node_bp`.** Shocks are defined on **named tenor pillars** (for example **6M** and **5Y**). With **`interpolate`**, the **basis-point bump** between two pillars varies **linearly** in tenor between those pillars; **at or beyond** the endpoints, the bump is usually held **flat** at the nearest pillar’s level (so the short end gets the 6M bump and the long end the 5Y bump in a two-node example). This is **not** the same as interpolating **discount factors** themselves—the scenario layer adjusts **quotes / zero rates** per engine rules, then rebuilds the curve. If a pillar lies **outside** the curve’s domain, expect **extrapolation warnings** and **flat** extension off the last live pillar.

In [ ]:
import json
from datetime import date

from finstack.scenarios import apply_scenario_to_market, build_scenario_spec
from finstack.core.market_data import DiscountCurve, ForwardCurve, MarketContext

AS_OF = "2025-01-15"
bd = date(2025, 1, 15)

# Fresh market: OIS discount + 3M forward curve (forward shock shown for illustration).
ois_knots = [(0.0, 1.0), (0.5, 0.994), (1.0, 0.985), (2.0, 0.965), (5.0, 0.88)]
mc = MarketContext()
mc.insert(DiscountCurve("USD-OIS", bd, ois_knots, day_count="act_365f"))
mc.insert(
    ForwardCurve(
        "USD-SOFR-3M",
        0.25,
        [(0.0, 0.045), (1.0, 0.048), (5.0, 0.052)],
        bd,
        day_count="act_360",
    )
)
base_json = mc.to_json()
print("Base market built; JSON length:", len(base_json))


def discount_from_market_json(mj: str, curve_id: str):
    data = json.loads(mj)
    for c in data.get("curves", []):
        if c.get("type") == "discount" and c.get("id") == curve_id:
            knots = [(float(t), float(df)) for t, df in c["knot_points"]]
            return DiscountCurve(
                c["id"],
                date.fromisoformat(c["base"]),
                knots,
                interp=c.get("interp_style", "monotone_convex").lower(),
                extrapolation=c.get("extrapolation", "flat_forward").lower().replace("_", "_"),
                day_count=c.get("day_count", "act_365f"),
            )
    raise ValueError(f"discount {curve_id} not found")


def print_df_pair(label: str, base_curve: DiscountCurve, stressed_mj: str):
    sc = discount_from_market_json(stressed_mj, base_curve.id)
    for t in (0.5, 1.0, 2.0, 5.0):
        b = base_curve.df(t)
        a = sc.df(t)
        print(f"  {label} t={t}Y  DF {b:.6f} -> {a:.6f}  (delta {a - b:+.6f})")


scenarios = [
    (
        "+100bp parallel",
        build_scenario_spec(
            "p100",
            json.dumps(
                [
                    {
                        "kind": "curve_parallel_bp",
                        "curve_kind": "discount",
                        "curve_id": "USD-OIS",
                        "bp": 100.0,
                    }
                ]
            ),
            "p",
            "d",
        ),
    ),
    (
        "Steepener",
        build_scenario_spec(
            "st",
            json.dumps(
                [
                    {
                        "kind": "curve_node_bp",
                        "curve_kind": "discount",
                        "curve_id": "USD-OIS",
                        "nodes": [["6M", 0.0], ["5Y", 60.0]],
                        "match_mode": "interpolate",
                    }
                ]
            ),
            "s",
            "d",
        ),
    ),
    (
        "Flattener",
        build_scenario_spec(
            "fl",
            json.dumps(
                [
                    {
                        "kind": "curve_node_bp",
                        "curve_kind": "discount",
                        "curve_id": "USD-OIS",
                        "nodes": [["6M", 55.0], ["5Y", 5.0]],
                        "match_mode": "interpolate",
                    }
                ]
            ),
            "f",
            "d",
        ),
    ),
    (
        "Inversion-style",
        build_scenario_spec(
            "inv",
            json.dumps(
                [
                    {
                        "kind": "curve_node_bp",
                        "curve_kind": "discount",
                        "curve_id": "USD-OIS",
                        "nodes": [["3M", 90.0], ["2Y", 40.0], ["5Y", -25.0]],
                        "match_mode": "interpolate",
                    }
                ]
            ),
            "i",
            "d",
        ),
    ),
]

base_dc = mc.get_discount("USD-OIS")
print("Base implied zero rates (curve.zero(t), same pillars as DF table):")
for t in (0.25, 0.5, 1.0, 2.0, 5.0):
    print(f"  t={t}Y  zero={base_dc.zero(t):.6f}")

for name, spec in scenarios:
    out = apply_scenario_to_market(spec, base_json, AS_OF)
    print(f"\n=== {name} ===")
    print("  operations_applied:", out["operations_applied"])
    print("  warnings:", out["warnings"])
    print_df_pair("USD-OIS", base_dc, out["market_json"])
    if name == "Steepener":
        sc = discount_from_market_json(out["market_json"], "USD-OIS")
        print("  Implied zero rates: base -> steepener (illustrates interpolate between 6M and 5Y nodes)")
        for t in (0.25, 0.5, 1.0, 2.0, 5.0):
            print(f"    t={t}Y  {base_dc.zero(t):.6f} -> {sc.zero(t):.6f}")

# Forward curve: small parallel bump (custom) to show ForwardCurve in the pipeline
fwd_spec = build_scenario_spec(
    "fwd_bump",
    json.dumps(
        [
            {
                "kind": "curve_parallel_bp",
                "curve_kind": "forward",
                "curve_id": "USD-SOFR-3M",
                "bp": 15.0,
            }
        ]
    ),
    "f",
    "d",
)
fr = apply_scenario_to_market(fwd_spec, base_json, AS_OF)
print("\n=== Forward +15bp parallel (USD-SOFR-3M) ===")
print("  operations_applied:", fr["operations_applied"])
for c in json.loads(fr["market_json"]).get("curves", []):
    if c.get("type") == "forward" and c.get("id") == "USD-SOFR-3M":
        k0 = c["knot_points"][0]
        print("  first knot forward rate after shock:", k0)
        break


## Takeaways

- **`curve_parallel_bp`** is the workhorse for parallel DV01-style shocks on **`discount`** or **`forward`** curves.
- **`curve_node_bp`** with **`match_mode: interpolate`** implements **key-rate** style shapes: steepener, flattener, and inversion-like twists.
- **`apply_scenario_to_market`** returns a stressed **`market_json`**; rebuild **`DiscountCurve`** objects (from JSON) to report **discount factors** before vs after.
- Built-in **historical templates** may include a dedicated rates pack—use **`try`/`except`** when template IDs differ by build.